In [1]:
import re
import unicodedata
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

def english_tokens(text):
    text = text.lower()
    return re.findall(r"[a-z]+", text)

def hindi_tokens(text):
    text = normalize_unicode(text)
    # Devanagari words; punctuation/numbers are excluded.
    return re.findall(r"[\u0900-\u097F]+", text)

def average_vectors(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

print("Core libraries loaded.")


Core libraries loaded.


In [2]:
# Q4(a): 20 aligned English-Hindi sentence pairs.
parallel_pairs = [('The machine learning model predicts future demand.', 'मशीन लर्निंग मॉडल भविष्य की मांग का अनुमान लगाता है।'), ('Deep learning networks classify images accurately.', 'डीप लर्निंग नेटवर्क चित्रों को सही ढंग से वर्गीकृत करते हैं।'), ('Natural language processing analyzes human language.', 'प्राकृतिक भाषा प्रसंस्करण मानव भाषा का विश्लेषण करता है।'), ('Artificial intelligence solves complex problems.', 'कृत्रिम बुद्धिमत्ता जटिल समस्याओं को हल करती है।'), ('Computer vision detects objects in images.', 'कंप्यूटर विजन चित्रों में वस्तुओं का पता लगाता है।'), ('Data science combines statistics and programming.', 'डेटा विज्ञान सांख्यिकी और प्रोग्रामिंग को जोड़ता है।'), ('Students learn programming in computer science.', 'छात्र कंप्यूटर विज्ञान में प्रोग्रामिंग सीखते हैं।'), ('Online education provides courses and exercises.', 'ऑनलाइन शिक्षा पाठ्यक्रम और अभ्यास प्रदान करती है।'), ('Teachers analyze student learning outcomes.', 'शिक्षक छात्र के सीखने के परिणामों का विश्लेषण करते हैं।'), ('Researchers train models using large datasets.', 'शोधकर्ता बड़े डेटासेट का उपयोग करके मॉडल प्रशिक्षित करते हैं।'), ('Machine learning improves prediction accuracy.', 'मशीन लर्निंग पूर्वानुमान की सटीकता में सुधार करती है।'), ('Neural networks learn useful patterns from data.', 'न्यूरल नेटवर्क डेटा से उपयोगी पैटर्न सीखते हैं।'), ('Language models learn patterns from text.', 'भाषा मॉडल पाठ से पैटर्न सीखते हैं।'), ('Image classification assigns labels to images.', 'चित्र वर्गीकरण चित्रों को लेबल प्रदान करता है।'), ('Algorithms process data to produce results.', 'एल्गोरिदम परिणाम उत्पन्न करने के लिए डेटा संसाधित करते हैं।'), ('Students practice algorithms and data structures.', 'छात्र एल्गोरिदम और डेटा संरचनाओं का अभ्यास करते हैं।'), ('Education platforms track student progress.', 'शिक्षा मंच छात्र की प्रगति को ट्रैक करते हैं।'), ('Data analysis helps researchers find patterns.', 'डेटा विश्लेषण शोधकर्ताओं को पैटर्न खोजने में मदद करता है।'), ('Prediction models estimate future outcomes.', 'पूर्वानुमान मॉडल भविष्य के परिणामों का अनुमान लगाते हैं।'), ('Technology systems use intelligent algorithms.', 'प्रौद्योगिकी प्रणालियां बुद्धिमान एल्गोरिदम का उपयोग करती हैं।')]
assert len(parallel_pairs) >= 20

english_sentences = [x[0] for x in parallel_pairs]
hindi_sentences = [x[1] for x in parallel_pairs]

print("Number of aligned pairs:", len(parallel_pairs))
for i in range(3):
    print(f"{i+1}. EN:", english_sentences[i])
    print("   HI:", hindi_sentences[i])


Number of aligned pairs: 20
1. EN: The machine learning model predicts future demand.
   HI: मशीन लर्निंग मॉडल भविष्य की मांग का अनुमान लगाता है।
2. EN: Deep learning networks classify images accurately.
   HI: डीप लर्निंग नेटवर्क चित्रों को सही ढंग से वर्गीकृत करते हैं।
3. EN: Natural language processing analyzes human language.
   HI: प्राकृतिक भाषा प्रसंस्करण मानव भाषा का विश्लेषण करता है।


In [3]:
# Q4(b): Preprocess both languages while preserving alignment.
english_tokenized = [english_tokens(s) for s in english_sentences]
hindi_tokenized = [hindi_tokens(s) for s in hindi_sentences]

print("English tokens:", english_tokenized[0])
print("Hindi tokens:", hindi_tokenized[0])


English tokens: ['the', 'machine', 'learning', 'model', 'predicts', 'future', 'demand']
Hindi tokens: ['मशीन', 'लर्निंग', 'मॉडल', 'भविष्य', 'की', 'मांग', 'का', 'अनुमान', 'लगाता', 'है।']


In [4]:
# Train monolingual Word2Vec models.
english_model = Word2Vec(
    sentences=english_tokenized,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    sg=0,
    epochs=150,
    seed=42
)

hindi_model = Word2Vec(
    sentences=hindi_tokenized,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    sg=0,
    epochs=150,
    seed=42
)

print("English dimension:", english_model.vector_size)
print("Hindi dimension:", hindi_model.vector_size)


English dimension: 50
Hindi dimension: 50


In [5]:
# Q4(c): Bilingual anchor-word pairs for alignment.
anchor_pairs = [
    ("machine", "मशीन"), ("learning", "लर्निंग"), ("model", "मॉडल"),
    ("data", "डेटा"), ("deep", "डीप"), ("neural", "न्यूरल"),
    ("network", "नेटवर्क"), ("language", "भाषा"), ("processing", "प्रसंस्करण"),
    ("image", "चित्र"), ("computer", "कंप्यूटर"), ("vision", "विजन"),
    ("student", "छात्र"), ("programming", "प्रोग्रामिंग"), ("education", "शिक्षा"),
    ("teacher", "शिक्षक"), ("researcher", "शोधकर्ता"), ("algorithm", "एल्गोरिदम"),
    ("prediction", "पूर्वानुमान")
]

valid_anchors = [
    (en, hi) for en, hi in anchor_pairs
    if en in english_model.wv and hi in hindi_model.wv
]

print("Valid bilingual anchors:", len(valid_anchors))


Valid bilingual anchors: 15


In [6]:
# Learn an orthogonal mapping from Hindi space to English space.
H = np.vstack([hindi_model.wv[hi] for en, hi in valid_anchors])
E = np.vstack([english_model.wv[en] for en, hi in valid_anchors])

U, _, Vt = np.linalg.svd(H.T @ E)
W = U @ Vt

print("Alignment matrix shape:", W.shape)
print("Hindi vectors are mapped into the English embedding space.")


Alignment matrix shape: (50, 50)
Hindi vectors are mapped into the English embedding space.


In [7]:
# Q4(c): Sentence representations using average word embeddings.
english_sentence_vectors = np.vstack([
    average_vectors(tokens, english_model)
    for tokens in english_tokenized
])

hindi_sentence_vectors = np.vstack([
    average_vectors(tokens, hindi_model)
    for tokens in hindi_tokenized
])

hindi_aligned_sentence_vectors = hindi_sentence_vectors @ W

print("English sentence-vector shape:", english_sentence_vectors.shape)
print("Hindi sentence-vector shape:", hindi_sentence_vectors.shape)
print("Aligned Hindi sentence-vector shape:", hindi_aligned_sentence_vectors.shape)


English sentence-vector shape: (20, 50)
Hindi sentence-vector shape: (20, 50)
Aligned Hindi sentence-vector shape: (20, 50)


In [8]:
# Q4(d/e): Rank Hindi candidates for at least five English queries.
query_indices = [0, 2, 4, 6, 9]

retrieval_rows = []
recall1_hits = 0
recall5_hits = 0

for qi in query_indices:
    query_vector = english_sentence_vectors[qi].reshape(1, -1)

    scores = cosine_similarity(
        query_vector,
        hindi_aligned_sentence_vectors
    ).flatten()

    ranking = np.argsort(-scores)
    rank_of_correct = int(np.where(ranking == qi)[0][0]) + 1

    if rank_of_correct == 1:
        recall1_hits += 1
    if rank_of_correct <= 5:
        recall5_hits += 1

    print(f"\nQUERY {qi+1}: {english_sentences[qi]}")
    for rank, idx in enumerate(ranking[:5], 1):
        print(f"Rank {rank}: score={scores[idx]:.4f} | {hindi_sentences[idx]}")

    retrieval_rows.append([
        qi + 1,
        english_sentences[qi],
        hindi_sentences[ranking[0]],
        float(scores[ranking[0]]),
        rank_of_correct
    ])

retrieval_table = pd.DataFrame(
    retrieval_rows,
    columns=[
        "Query #", "English Query", "Rank-1 Retrieved Hindi",
        "Rank-1 Similarity", "Correct Translation Rank"
    ]
)

display(retrieval_table.round(4))



QUERY 1: The machine learning model predicts future demand.
Rank 1: score=0.8807 | मशीन लर्निंग मॉडल भविष्य की मांग का अनुमान लगाता है।
Rank 2: score=0.8780 | चित्र वर्गीकरण चित्रों को लेबल प्रदान करता है।
Rank 3: score=0.8772 | मशीन लर्निंग पूर्वानुमान की सटीकता में सुधार करती है।
Rank 4: score=0.8771 | डीप लर्निंग नेटवर्क चित्रों को सही ढंग से वर्गीकृत करते हैं।
Rank 5: score=0.8769 | शिक्षक छात्र के सीखने के परिणामों का विश्लेषण करते हैं।

QUERY 3: Natural language processing analyzes human language.
Rank 1: score=0.8215 | प्राकृतिक भाषा प्रसंस्करण मानव भाषा का विश्लेषण करता है।
Rank 2: score=0.8156 | भाषा मॉडल पाठ से पैटर्न सीखते हैं।
Rank 3: score=0.8060 | डेटा विश्लेषण शोधकर्ताओं को पैटर्न खोजने में मदद करता है।
Rank 4: score=0.8050 | शोधकर्ता बड़े डेटासेट का उपयोग करके मॉडल प्रशिक्षित करते हैं।
Rank 5: score=0.8036 | कृत्रिम बुद्धिमत्ता जटिल समस्याओं को हल करती है।

QUERY 5: Computer vision detects objects in images.
Rank 1: score=0.7143 | छात्र कंप्यूटर विज्ञान में प्रोग्रामिं

,Query #,English Query,Rank-1 Retrieved Hindi,Rank-1 Similarity,Correct Translation Rank
0,1,The machine learning model predicts future dem...,मशीन लर्निंग मॉडल भविष्य की मांग का अनुमान लगा...,0.8807,1
1,3,Natural language processing analyzes human lan...,प्राकृतिक भाषा प्रसंस्करण मानव भाषा का विश्लेष...,0.8215,1
2,5,Computer vision detects objects in images.,छात्र कंप्यूटर विज्ञान में प्रोग्रामिंग सीखते ...,0.7143,2
3,7,Students learn programming in computer science.,छात्र कंप्यूटर विज्ञान में प्रोग्रामिंग सीखते ...,0.7763,1
4,10,Researchers train models using large datasets.,मशीन लर्निंग पूर्वानुमान की सटीकता में सुधार क...,0.8435,19


In [9]:
# Q4(f/g): Recall@1 and Recall@5.
n_queries = len(query_indices)
recall_at_1 = recall1_hits / n_queries
recall_at_5 = recall5_hits / n_queries

print(f"Recall@1: {recall_at_1:.4f} ({recall1_hits}/{n_queries})")
print(f"Recall@5: {recall5_hits:.4f} ({recall5_hits}/{n_queries})")


Recall@1: 0.6000 (3/5)
Recall@5: 4.0000 (4/5)


### Q4(i): Cross-lingual retrieval challenges

If English and Hindi embeddings are trained independently, their vector spaces are not automatically aligned. The same concept can occupy unrelated coordinates in the two spaces.

A multilingual or aligned embedding approach addresses this by learning a shared space or a mapping between spaces. This notebook demonstrates a simple **orthogonal bilingual alignment** using corresponding English–Hindi anchor words.

Other challenges include morphology, word-order differences, limited vocabulary, translation ambiguity, out-of-vocabulary words, differences in corpus size/quality, and imperfect sentence-level averaging.

**Recall@1** measures whether the correct aligned sentence is ranked first. **Recall@5** measures whether it appears among the top five candidates.


### Final observation

Cross-lingual retrieval requires comparable representations across languages. Monolingual embeddings alone are not sufficient because their coordinate systems are independently learned. Alignment or multilingual embeddings make cosine-similarity retrieval between languages possible.
